# Stage C — bounded genomic pilot
Trains separately initialized adaptive, reference, frozen-memory, and no-memory runs. Re-running resumes each run from Drive.

In [ ]:
# USER CONFIGURATION
REPO_URL='https://github.com/Gonza10V/SeqTrainer.git'
GIT_REF='9a17bf2953323d51fcec7b9d90d1a2aa2270bc66'
DRIVE_ROOT='/content/drive/MyDrive/SeqTrainerStageC'
# This is a numerical-stability calibration. Use a new name for every distinct configuration.
PILOT_NAME='c7_calibration_50k_lr3e5'
HORIZON=3
VALID_BASE_BUDGET=50_000
CHECKPOINT_EVERY=25
LEARNING_RATE=3e-5
GRADIENT_CLIP_NORM=0.5
VALIDATION_STREAMS=4

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, subprocess,sys
repo=Path('/content/SeqTrainer')
if not repo.exists(): subprocess.run(['git','clone',REPO_URL,str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'remote','set-url','origin',REPO_URL],check=True)
subprocess.run(['git','-C',str(repo),'fetch','--all'],check=True)
subprocess.run(['git','-C',str(repo),'checkout',GIT_REF],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',f'{repo}[torch,bacteria-titan]'],check=True)
selection_path=Path(DRIVE_ROOT)/'runs'/'c1_tokenizers_cpu'/'tokenizer_selection.json'
selection=json.loads(selection_path.read_text(encoding='utf-8'))
selected=selection.get('selected_tokenizer')
if not isinstance(selected,str) or not selected: raise ValueError('tokenizer_selection.json has no selected_tokenizer')
dataset=Path(DRIVE_ROOT)/'stage_c_dataset'/'ordered_streams'/selected
if not (dataset/'token_stream_manifest.json').is_file(): raise FileNotFoundError(f'Run Notebook 00b first; missing {dataset}/token_stream_manifest.json')
DATASET_DIR=str(dataset)
print('Resolved Handoff 00b dataset:',DATASET_DIR)
root=f'{DRIVE_ROOT}/runs/{PILOT_NAME}'
subprocess.run(['seqtrainer-titans-stage-c-colab-run','--run-dir',root,'--label','hardware_preflight','--repo',str(repo),'--','seqtrainer-titans-stage-c-hardware-preflight','--require','A100'],check=True)

In [ ]:
def show_failure(run_dir, label):
    failure=Path(run_dir)/'FAILED.txt'
    log=Path(run_dir)/'logs'/f'{label}.log'
    if failure.exists(): print(failure.read_text(encoding='utf-8',errors='replace'))
    if log.exists():
        print(f'--- tail of {log} ---')
        print(log.read_text(encoding='utf-8',errors='replace')[-12000:])

for mode in ['adaptive','reference','frozen_memory','no_memory']:
    run_dir=f'{root}/{mode}'
    label=f'train_{mode}'
    command=['seqtrainer-titans-stage-c-train','--dataset-dir',DATASET_DIR,'--run-dir',run_dir,'--memory-mode',mode,'--horizon',str(HORIZON),'--batch-size','1','--max-valid-bases',str(VALID_BASE_BUDGET),'--checkpoint-every',str(CHECKPOINT_EVERY),'--learning-rate',str(LEARNING_RATE),'--gradient-clip-norm',str(GRADIENT_CLIP_NORM),'--validation-streams',str(VALIDATION_STREAMS),'--activation','float32']
    try:
        subprocess.run(['seqtrainer-titans-stage-c-colab-run','--run-dir',run_dir,'--label',label,'--repo',str(repo),'--',*command],check=True)
    except subprocess.CalledProcessError:
        show_failure(run_dir,label)
        raise
print('SHARE THIS DIRECTORY:',root)
print('Live status during training: <run>/<mode>/LIVE_STATUS.json')